In [3]:
from devwer import Metrics

metrics = Metrics()

print('WER:')
print(metrics.wer("संगम","सङ्गम"))
print(metrics.wer('यह एक परीक्षण है।', 'यह एक परीक्षण है'))
print(metrics.wer("संगम","सङ्गम"))

print('CER:')
print(metrics.cer("संगम","सङ्गम"))
print(metrics.cer('यह एक परीक्षण है।', 'यह एक परीक्षण है'))
print(metrics.cer("संगम","सङ्गम"))

print(metrics.wer_legacy('यह एक परीक्षण है।', 'यह एक परीक्षण है'))
print(metrics.tokenize('cer','यह एक परीक्षण है।'))



WER:
0.0
0.0
0.0
CER:
0.0
0.0
0.0
0.0
['य', 'ह', ' ', 'ए', 'क', ' ', 'प', 'र', 'ि', 'क', '्', 'ष', 'ण', ' ', 'ह', 'ै']


In [76]:
import re

consonants = 'कखगघङचछजझञटठडढणतथदधनपफबभमयरलवशषसह'
vowel_signs = {
            'ि': 'ि',
            'ी': 'ी',
            'ु': 'ु',
            'ू': 'ू',
            'ृ': 'ृ',
            'े': 'े',
            'ै': 'ै',
            'ो': 'ो',
            'ौ': 'ौ',
            'ं': 'ं'
        }

complex_characters = {
    'त्व': 'त_्_व',
    'ण्ढ': 'ण_्_ढ',
    'स्थ': 'स_्_थ',
    'श्व': 'श_्_व',
    'श्न': 'श_्_न',
    'श्च': 'श_्_च',
    'श्ल': 'श_्_ल',
    'श्र': 'श_्_र',
    'शृ': 'श_ृ्_',
    'र्व': 'र_्_व',
    'र्वा': 'र_्_वा',
    'र्स्प': 'र_्_स्प',
    'ट्र': 'ट_्_र',
    'ठ्र': 'ठ_्_र',
    'ड्र': 'ड_्_र',
    'ढ्र': 'ढ_्_र',
    'छ्र': 'छ_्_र',
    'क्र': 'क_्_र',
    'ग्र': 'ग_्_र',
    'भ्र': 'भ_्_र',
    'ब्र': 'ब_्_र',
    'क्ष': 'क_्_ष',
    'ज्ञ': 'ज_्_ञ'
}

rcomplex_characters = {v: k for k, v in complex_characters.items()}

def replace_decomposed_with_original(text):
    # Replace decomposed forms with original characters
    for decomposed, original in rcomplex_characters.items():
        text = re.sub(re.escape(decomposed), original, text)
    return text

def replacer(match):
    consonant = match.group(1)
    half_vowel = match.group(2)
    key = f'{consonant}{half_vowel}'
    return key

def replace_const_half_vowel(text):
    pattern = f"([{consonants}]_्_[{''.join(re.escape(vowel) for vowel in vowel_signs.keys())}])"
    return re.sub(pattern, replacer, text)

sentence = 'यह एक परीक्षण है।'
# sentence = list(sentence.replace(' ',''))

# naya = ('_'.join(sentence))
naya = sentence
print(naya)

naya = replace_decomposed_with_original(naya)
print(naya)

naya = replace_const_half_vowel(naya)
print(naya)

यह एक परीक्षण है।
यह एक परीक्षण है।
यह एक परीक्षण है।


In [85]:
import re

# Define regex patterns for complex conjuncts, consonant + vowel combinations, and individual characters
def build_patterns():
    # List of complex conjuncts
    complex_conjuncts = 'ज्ञ|क्ष|त्र|श्र|स्र|प्र|ब्र|फ्र|ग्र|ह्र'
    # List of consonants, vowels, and half-vowels
    consonants = 'कगघचछजझटठडढणतथदधनपफबभमयरलवशषसह'
    vowels = 'अआइईउऊएऐओऔ'
    half_vowels = 'िुूेै'
    # Combine patterns for complex conjuncts, consonant + vowel combinations, consonant + half-vowel, and individual characters
    complex_pattern = f'({complex_conjuncts})'
    consonant_vowel_pattern = f'([{consonants}])([{vowels}])'
    consonant_half_vowel_pattern = f'([{consonants}])([{half_vowels}])'
    individual_pattern = f'([{consonants}]|[{vowels}]|[{half_vowels}])'
    return complex_pattern, consonant_vowel_pattern, consonant_half_vowel_pattern, individual_pattern

# Function to split text into fragments while preserving complex conjuncts and consonant + vowel combinations
def split_text_into_fragments(text):
    complex_pattern, consonant_vowel_pattern, consonant_half_vowel_pattern, individual_pattern = build_patterns()

    # Combine patterns to match complex conjuncts, consonant + vowel combinations, consonant + half-vowel, and individual characters
    combined_pattern = f'({complex_pattern}|{consonant_vowel_pattern}|{consonant_half_vowel_pattern})'

    # Find all matches for complex conjuncts, consonant + vowel combinations, and consonant + half-vowel combinations
    matches = re.findall(combined_pattern, text)

    # Flatten the list of tuples and filter out empty strings
    fragments = [item for sublist in matches for item in sublist if item]

    # Process remaining characters which were not captured in the above patterns
    remaining_text = re.sub(combined_pattern, '', text)
    remaining_fragments = list(remaining_text)

    # Combine and deduplicate the fragments
    all_fragments = fragments + remaining_fragments
    unique_fragments = list(dict.fromkeys(all_fragments))

    return unique_fragments

# Example input text with consonant + vowel combinations, complex conjuncts, and individual characters
input_text = 'ज्ञानेक्षत्र प्रेयश्चश्रण कुतु'

# Split the text into fragments
fragments = split_text_into_fragments(input_text)
print(fragments)

['ज्ञ', 'ने', 'न', 'े', 'क्ष', 'त्र', 'प्र', 'श्र', 'कु', 'क', 'ु', 'तु', 'त', 'ा', ' ', 'य', 'श', '्', 'च', 'ण']


Combined Sentence: क् ष
Decomposed Sentence: क् ष
